In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# ---------------- Load data ----------------
gdf = gpd.read_file(r"C:\Users\Anurodh\Downloads\Hyderabad\stop_hexagon_normalized_morphology_rings_demand.gpkg")

rings = ['r1', 'r2', 'r3']
vars_ = ['bldg_cnt','BCR','FAR','OSR','BVD','median_h','mean_h','VHI']

base_out_dir = r"C:\Users\Anurodh\Downloads\Hyderabad\RandomForest_LogDemand"
os.makedirs(base_out_dir, exist_ok=True)

results = {}

# =========================================================
# STEP 1: MODEL + SHAP + METRICS
# =========================================================
rf_stats = []

for ring in rings:

    print(f"\nProcessing {ring}...")

    X = gdf[[f'{ring}_{v}' for v in vars_]]
    y = gdf['Log_Demand']

    data = pd.concat([X, y], axis=1).dropna()
    X_clean = data[X.columns]
    y_clean = data['Log_Demand']

    # ---- train-test split for stats ----
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(n_estimators=500, random_state=42)
    model.fit(X_train, y_train)

    # ---- predictions ----
    y_pred = model.predict(X_test)

    # ---- metrics ----
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    rf_stats.append([ring, r2, rmse, mae])

    # ---- SHAP ----
    # Train model on FULL data for SHAP consistency
    model_full = RandomForestRegressor(n_estimators=500, random_state=42)
    model_full.fit(X_clean, y_clean)
    
    explainer = shap.Explainer(model_full, X_clean)
    shap_values = explainer(X_clean)

    results[ring] = {
        "X": X_clean,
        "model": model_full,   # use full model everywhere
        "shap": shap_values.values
    }

# ---- Save RF stats ----
rf_df = pd.DataFrame(rf_stats, columns=["Ring", "R2", "RMSE", "MAE"])
rf_df.to_csv(os.path.join(base_out_dir, "RF_Model_Stats.csv"), index=False)

print("\nRandom Forest Stats:")
print(rf_df)

# =========================================================
# STEP 2: PERMUTATION IMPORTANCE
# =========================================================
perm_dir = os.path.join(base_out_dir, "Permutation")
os.makedirs(perm_dir, exist_ok=True)

for ring in rings:

    X = results[ring]["X"]
    model = results[ring]["model"]

    y = gdf.loc[X.index, 'Demand']

    perm = permutation_importance(model, X, y, n_repeats=10, random_state=42)

    perm_df = pd.DataFrame({
        "feature": X.columns,
        "importance": perm.importances_mean
    }).sort_values(by="importance", ascending=False)

    perm_df.to_csv(os.path.join(perm_dir, f"Permutation_{ring}.csv"), index=False)

# =========================================================
# STEP 3–5: PDP + ELASTICITY + THRESHOLD DETECTION
# =========================================================
pdp_dir = os.path.join(base_out_dir, "PDP_Elasticity")
os.makedirs(pdp_dir, exist_ok=True)

threshold_results = []

for ring in rings:

    print(f"\nAnalyzing PDP + Elasticity for {ring}...")

    X = results[ring]["X"]
    model = results[ring]["model"]
    shap_values = results[ring]["shap"]

    # ---- top 3 features ----
    mean_shap = np.abs(shap_values).mean(axis=0)
    top_idx = np.argsort(mean_shap)[::-1][:3]
    top_features = X.columns[top_idx]

    fig, axes = plt.subplots(3, 3, figsize=(15, 12))

    for i, feat in enumerate(top_features):

        # ---------------- PDP ----------------
        pdp_disp = PartialDependenceDisplay.from_estimator(
            model,
            X,
            [feat],
            ax=axes[i, 0],
            kind="both",
            grid_resolution=50
        )
        
        axes[i, 0].set_title(f"PDP + ICE: {feat}")
        
        # ---- SAFE extraction (handles sklearn >= 1.2 key changes) ----
        pdp_data = pdp_disp.pd_results[0]

        # Try new key names first, fall back to old ones
        if "grid_values" in pdp_data:
            grid = pdp_data["grid_values"][0]
        elif "values" in pdp_data:
            grid = pdp_data["values"][0]
        else:
            raise KeyError(f"Cannot find grid key. Available keys: {list(pdp_data.keys())}")

        if "average" in pdp_data:
            values = pdp_data["average"][0]
        elif "individual" in pdp_data:
            values = pdp_data["individual"][0].mean(axis=0)
        else:
            raise KeyError(f"Cannot find values key. Available keys: {list(pdp_data.keys())}")
        
        # ---------------- ELASTICITY ----------------
        dy = np.gradient(values)
        dx = np.gradient(grid)
        elasticity = (dy / (dx + 1e-9)) * (grid / (values + 1e-9))
        
        axes[i, 1].plot(grid, elasticity)
        axes[i, 1].axhline(0, linestyle="--")
        axes[i, 1].set_title(f"Elasticity: {feat}")
        
        # ---------------- THRESHOLD ----------------
        d2 = np.gradient(dy)
        threshold_idx = np.argmax(np.abs(d2))
        threshold_val = grid[threshold_idx]
        
        threshold_results.append([ring, feat, threshold_val])
        
        axes[i, 2].plot(grid, values)
        axes[i, 2].axvline(threshold_val, color='red', linestyle='--')
        axes[i, 2].set_title(f"Threshold: {feat}")

    plt.tight_layout()
    plt.savefig(os.path.join(pdp_dir, f"PDP_Elasticity_{ring}.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

# ---- Save thresholds ----
threshold_df = pd.DataFrame(threshold_results,
                            columns=["Ring", "Feature", "Threshold"])
threshold_df.to_csv(os.path.join(base_out_dir, "Thresholds.csv"), index=False)

print("\nThresholds detected:")
print(threshold_df)

print("\nAll analysis complete.")

1. Compute SHAP values (consistent feature attribution)
   - Train Random Forest model
   - Use SHAP TreeExplainer
   - Identify top predictors per ring

2. Validate feature importance using Permutation Importance
   - Apply sklearn permutation_importance
   - Compare rankings with SHAP outputs
   - Ensure consistency and robustness

3. Generate PDP (Partial Dependence Plots) and ICE (Individual Conditional Expectation)
   - Compute PDP for top predictors
   - Generate ICE curves for heterogeneity analysis
   - Separate results for Ring 1 and Ring 3

4. Convert PDP into Elasticity Curves
   - Compute numerical derivative of PDP
   - Normalize to obtain elasticity:
     Elasticity = (dY/dX) * (X/Y)
   - Interpret marginal sensitivity of demand

5. Detect Threshold Values (FAR / BCR)
   - Identify inflection points in PDP / elasticity curves
   - Detect saturation or diminishing returns zones
   - Extract critical FAR and BCR thresholds

6. Output Random Forest Diagnostics per Ring
   - R² score
   - RMSE / MAE
   - Feature importance rankings
   - Model stability checks

7. Save Outputs in Structured Format
   - SHAP values → CSV / plots
   - PDP & ICE plots → images
   - Elasticity curves → CSV
   - Threshold values → summary table
   - Model diagnostics → report file

# Hyderabad Metro Morphology–Ridership Analysis  
*(Random Forest + PDP/ICE + Elasticity + Thresholds)*

---

## 1. Model Performance & Spatial Signal

| Ring | R² | RMSE | MAE | Interpretation |
|------|----|------|-----|---------------|
| r1 | 0.272 | 0.609 | 0.455 | Moderate explanatory power → strong station-area effect |
| r2 | 0.093 | 0.680 | 0.497 | Weak → transitional influence |
| r3 | 0.022 | 0.707 | 0.483 | Negligible → outer morphology weak |

**Insight:**  
Hyderabad exhibits a **strong distance-decay effect**, where morphology impacts ridership primarily within the **immediate station area (r1)**.

---

## 2. Key Morphological Drivers

Across all rings:

- **VHI (Vertical Heterogeneity Index)** → dominant nonlinear driver  
- **Building Count** → density/congestion proxy  
- **BCR / FAR** → built form intensity & compactness  

---

## 3. Threshold Summary

| Ring | Feature | Threshold |
|------|--------|----------|
| r1 | VHI | 0.268 |
| r1 | Building Count | 0.682 |
| r1 | BCR | 0.870 |
| r2 | VHI | 0.203 |
| r2 | Building Count | 0.669 |
| r2 | BCR | 0.427 |
| r3 | VHI | 0.171 |
| r3 | Building Count | 0.513 |
| r3 | FAR | 0.260 |

---

## 4. Ring-wise Interpretation

---

### 🔵 R1 — Immediate Station Area (Primary Zone)

**Key mechanisms:**

#### VHI (≈ 0.27)
- Sharp increase in ridership after threshold  
- Strong positive elasticity spike  

**Interpretation:**  
Minimum vertical diversity is required to activate demand.  
This is a **critical mass effect**, not gradual.

---

#### Building Count (≈ 0.68)
- Flat → then sharply negative  

**Interpretation:**  
Beyond threshold, **overcrowding reduces accessibility efficiency**.

---

#### BCR (≈ 0.87)
- Weak initially → strong positive at high values  

**Interpretation:**  
Highly compact built form improves ridership **only at very high consolidation**.

---

**R1 Summary:**

> Optimal station morphology =  
> - High vertical diversity  
> - Moderate density  
> - High but organized compactness  

---

### 🟡 R2 — Intermediate Zone (Transition)

#### VHI (≈ 0.20)
- Similar shape as r1 but weaker  

**Interpretation:**  
Diversity matters, but influence declines with distance.

---

#### Building Count (≈ 0.67)
- Consistent negative trend  

**Interpretation:**  
Density becomes **purely detrimental beyond moderate levels**.

---

#### BCR (≈ 0.43)
- U-shaped response  

**Interpretation:**  
Mid-range built form is inefficient.  
Either low-density or high-compact form works better.

---

**R2 Summary:**

> Transitional zone penalizes uncontrolled density  
> and favors structured morphology.

---

### 🔴 R3 — Outer Zone (Marginal Influence)

#### VHI (≈ 0.17)
- Small positive spike  

**Interpretation:**  
Effect exists structurally but is weak.

---

#### Building Count (≈ 0.51)
- Strong negative beyond threshold  

**Interpretation:**  
Outer density leads to **dispersion penalty**.

---

#### FAR (≈ 0.26)
- Inverted-U relationship  

**Interpretation:**  
Moderate FAR supports feeder demand;  
high FAR creates **self-contained zones**, reducing metro use.

---

**R3 Summary:**

> Outer morphology has limited impact and can become negative if overbuilt.

---

## 5. Cross-Ring Structural Patterns

### 1. Threshold Decay with Distance

| Variable | r1 | r2 | r3 | Pattern |
|----------|----|----|----|--------|
| VHI | 0.27 | 0.20 | 0.17 | Decreasing outward |
| Building Count | 0.68 | 0.67 | 0.51 | Decreasing outward |
| Intensity (BCR/FAR) | High | Medium | Low | Decreasing outward |

**Interpretation:**  
Closer to stations → higher morphological complexity tolerated.

---

### 2. Strong Nonlinearity

- Step changes in PDP  
- Sharp elasticity spikes  
- Clear thresholds  

**Implication:**  
Linear planning assumptions are invalid.

---

### 3. Density vs Efficiency Trade-off

- Low–moderate density → beneficial  
- High density → negative  

**Conclusion:**  
Hyderabad shows **congestion-driven morphology**, not capacity-driven.

---

## 6. Planning Implications

### ✔ Effective Strategies
- Promote vertical diversity (mixed-use + varied heights)
- Maintain moderate density near stations
- Encourage compact but organized development

### ❌ Ineffective Strategies
- Blind densification
- High FAR without structure
- Excess building fragmentation

---

## 7. Final Synthesis

> Hyderabad Metro ridership is governed by a  
> **threshold-based morphological system**, where:
>
> - Demand activates only after crossing minimum diversity thresholds  
> - Over-densification reduces efficiency  
> - Morphological influence sharply declines beyond the first ring  

---

## 8. One-Line Insight

> Hyderabad is a **“threshold TOD city”**, not a density-driven one —  
> ridership depends on *how* the built environment is structured, not just *how much* is built.

# Chennai Metro Morphology–Ridership Analysis  
*(Random Forest + PDP/ICE + Elasticity + Thresholds)*

---

## 1. Model Performance & Reliability

| Ring | R² | RMSE | MAE | Interpretation |
|------|----|------|-----|---------------|
| r1 | -1.562 | 0.715 | 0.581 | Model fails → worse than mean prediction |
| r2 | -0.780 | 0.596 | 0.433 | Very weak signal |
| r3 | -1.154 | 0.656 | 0.505 | No explanatory power |

**Critical Insight:**  
Unlike Hyderabad, Chennai shows **no stable predictive relationship** between morphology and ridership.

👉 This is not just weak — it is **structurally unstable modeling behavior**.

---

## 2. Dominant Variables (Observed, but Unreliable)

From PDP + elasticity:

- **OSR (Open Space Ratio)** → step-like effect at near-zero  
- **VHI (Vertical Heterogeneity)** → only at very high values  
- **Building Count** → mostly negative or flat  

⚠️ However, due to negative R², these patterns must be treated as:
> **indicative but not causally reliable**

---

## 3. Threshold Summary

| Ring | Feature | Threshold |
|------|--------|----------|
| r1 | OSR | 0.0095 |
| r1 | VHI | 0.842 |
| r1 | Building Count | 1.000 |
| r2 | OSR | 0.0003 |
| r2 | VHI | 0.718 |
| r2 | Building Count | 0.165 |
| r3 | VHI | 0.720 |
| r3 | Building Count | 0.176 |
| r3 | OSR | 0.0002 |

---

## 4. Ring-wise Interpretation

---

### 🔵 R1 — Immediate Station Area (Unstable Core)

#### OSR (~0.01)
- Sharp jump from near-zero  
- Then completely flat  

**Interpretation:**  
Even minimal open space changes model output,  
but effect saturates instantly.

👉 Indicates **data sparsity / boundary artifact**, not real urban behavior.

---

#### VHI (~0.84)
- Effect only at extremely high values  
- Most range is flat  

**Interpretation:**  
Vertical diversity only matters in rare extreme cases.

---

#### Building Count (~1.0)
- Threshold at upper bound  

**Interpretation:**  
No meaningful variation → **model cannot learn density effects**

---

**R1 Summary:**

> No interpretable structure.  
> Morphology does not systematically explain ridership.

---

### 🟡 R2 — Intermediate Zone (Weak / Noisy Signal)

#### OSR (~0.0003)
- Immediate jump → flat  

**Interpretation:**  
Binary-like behavior (presence vs absence), not continuous influence.

---

#### VHI (~0.72)
- Slight increase after threshold  
- Weak elasticity  

**Interpretation:**  
Only very high diversity shows marginal effect.

---

#### Building Count (~0.16)
- Early drop, then flat  

**Interpretation:**  
Even small increases in density reduce predicted ridership,  
but effect is inconsistent.

---

**R2 Summary:**

> Weak, noisy relationships with no stable gradients.

---

### 🔴 R3 — Outer Zone (No Meaningful Influence)

#### VHI (~0.72)
- Minor positive shift  

#### Building Count (~0.18)
- Negative trend  

#### OSR (~0.00015)
- Instant saturation  

**Interpretation:**  
All variables show **near-zero elasticity across most range**.

---

**R3 Summary:**

> Outer morphology is effectively irrelevant.

---

## 5. Cross-Ring Structural Patterns

---

### 1. Extreme Threshold Behavior

- OSR thresholds ≈ 0  
- VHI thresholds ≈ very high (~0.7–0.85)  
- Building thresholds either very low or maxed out  

**Interpretation:**

> Model detects **edge conditions**, not continuous relationships.

---

### 2. Flat PDP + Near-zero Elasticity

- Most curves are flat  
- Elasticity ≈ 0 across ranges  

**Implication:**

> Morphological variables do not explain variation in demand.

---

### 3. Absence of Distance Decay Structure

Unlike Hyderabad:
- No systematic change across rings  
- No consistent threshold cascade  

---

## 6. Key Structural Diagnosis

### ❗ 1. Morphology–Demand Decoupling

> Chennai ridership is **not driven by built form variables used here**.

Possible reasons:
- Strong dependence on **network structure (interchanges, line hierarchy)**
- **Socio-economic dominance** (income, job distribution)
- **First/last-mile constraints**
- High **informal transport competition**

---

### ❗ 2. Data / Feature Limitation

Signals suggest:
- Low variance in key variables  
- Poor feature representativeness  
- Possible spatial aggregation issues  

---

### ❗ 3. Model Breakdown Indicator

Negative R² across all rings implies:

> The system is either:
> - Non-morphological, or  
> - Missing critical explanatory variables  

---

## 7. Planning Implications

### ✔ What this suggests:
- Built form interventions alone will **not significantly shift ridership**
- Focus should shift to:
  - Network connectivity  
  - Multimodal integration  
  - Accessibility improvements  

---

### ❌ What does NOT work:
- Density-based TOD assumptions  
- FAR/BCR-driven policies  
- Morphology-only planning frameworks  

---

## 8. Final Synthesis

> Chennai Metro exhibits a **non-morphological ridership system**, where:

- Built environment variables show **no stable or predictive influence**  
- Thresholds represent **data artifacts rather than behavioral shifts**  
- Spatial structure is likely governed by **network and socio-economic factors**

---

## 9. One-Line Insight

> Chennai is a **“network-driven transit system”**, not a morphology-driven one —  
> ridership depends more on connectivity than built form.

# Kochi Metro Morphology–Ridership Analysis  
*(Random Forest + PDP/ICE + Elasticity + Thresholds)*

---

## 1. Model Performance & Reliability

| Ring | R² | RMSE | MAE | Interpretation |
|------|----|------|-----|---------------|
| r1 | -4.586 | 2.322 | 2.142 | Severe model failure |
| r2 | -9.511 | 3.186 | 2.947 | Extremely unstable |
| r3 | -6.979 | 2.776 | 2.674 | No predictive structure |

**Critical Insight:**  
Kochi exhibits **complete model breakdown across all rings**.

👉 This is not just weak explanatory power — it indicates:  
> **Morphology variables are fundamentally misaligned with ridership behavior**

---

## 2. Dominant Variables (Observed but Structurally Weak)

From PDP + elasticity:

- **Building Count** → dominant but strongly negative  
- **OSR (Open Space Ratio)** → thresholded step behavior  
- **Height metrics (mean/median)** → unstable nonlinear spikes  
- **VHI** → minor effect only at very low threshold  

⚠️ Due to extreme negative R²:  
> These are **mathematical artifacts, not reliable causal drivers**

---

## 3. Threshold Summary

| Ring | Feature | Threshold |
|------|--------|----------|
| r1 | Building Count | 0.493 |
| r1 | OSR | 0.254 |
| r1 | VHI | 0.114 |
| r2 | Building Count | 0.947 |
| r2 | BCR | 0.443 |
| r2 | Mean Height | 0.456 |
| r3 | Median Height | 0.772 |
| r3 | Building Count | 0.837 |
| r3 | Mean Height | 0.427 |

---

## 4. Ring-wise Interpretation

---

### 🔵 R1 — Immediate Station Area (Highly Unstable Core)

#### Building Count (~0.49)
- Sharp drop beyond threshold  
- Strong negative elasticity spike  

**Interpretation:**  
Density quickly becomes detrimental.

👉 However:
> Effect is exaggerated and unstable → likely **overfitting artifact**

---

#### OSR (~0.25)
- Step change followed by flat response  

**Interpretation:**  
Open space shows **binary influence**, not continuous effect.

---

#### VHI (~0.11)
- Very low threshold  
- Minimal variation after  

**Interpretation:**  
Vertical diversity plays **almost no role**

---

**R1 Summary:**

> No coherent morphology–ridership mechanism  
> Only unstable threshold effects

---

### 🟡 R2 — Intermediate Zone (Extreme Nonlinearity)

#### Building Count (~0.95)
- Threshold at extreme upper bound  

**Interpretation:**  
Model only reacts at **edge cases**

---

#### BCR (~0.44)
- Sharp dip followed by recovery  

**Interpretation:**  
Mid-density built form appears inefficient,  
but pattern is inconsistent.

---

#### Mean Height (~0.46)
- Strong oscillations in elasticity  

**Interpretation:**  
Height influence is **erratic and non-systematic**

---

**R2 Summary:**

> Relationships are **non-monotonic, unstable, and non-interpretable**

---

### 🔴 R3 — Outer Zone (Noise-Dominated Behavior)

#### Median Height (~0.77)
- Sudden drop after threshold  

#### Building Count (~0.84)
- Negative effect at high values  

#### Mean Height (~0.43)
- Oscillatory behavior  

**Interpretation:**  
All variables show:
- Sharp discontinuities  
- Large elasticity spikes  
- No consistent direction  

---

**R3 Summary:**

> Outer zone is **pure noise from model perspective**

---

## 5. Cross-Ring Structural Patterns

---

### 1. Thresholds at Extremes

- Many thresholds near **upper bounds (~0.8–1.0)**  
- Others at **very low values (~0.1–0.25)**  

**Interpretation:**

> Model is reacting to **data boundaries**, not real urban transitions

---

### 2. High Elasticity Volatility

- Large spikes (positive & negative)  
- No smooth gradients  

**Implication:**

> System lacks stable response function

---

### 3. Absence of Spatial Logic

Unlike Hyderabad:
- No distance-decay  
- No consistent variable behavior  
- No structural similarity across rings  

---

## 6. Key Structural Diagnosis

---

### ❗ 1. Morphology is NOT the governing system

> Kochi ridership is largely **independent of built form variables used**

---

### ❗ 2. Strong External Drivers Likely Dominate

More plausible determinants:
- **Network coverage limitations**
- **Water-based geography constraints**
- **First/last-mile dependency**
- **Low urban intensity around stations**
- **Modal competition (ferry, bus, informal transport)**

---

### ❗ 3. Data + Scale Mismatch

Indications:
- Small sample size  
- Low morphological variation  
- Spatial aggregation masking effects  

---

### ❗ 4. Severe Model Instability

Negative R² of this magnitude implies:

> Model is effectively **fitting noise**, not signal

---

## 7. Planning Implications

---

### ✔ What this suggests:
- Morphology-based TOD strategies will have **minimal impact**
- Priority should shift to:
  - Network expansion  
  - Multimodal integration  
  - Accessibility improvements  

---

### ❌ What does NOT work:
- Density increases (building count)  
- Height manipulation strategies  
- FAR/BCR-based zoning interventions  

---

## 8. Final Synthesis

> Kochi Metro represents a **non-morphological transit system**, where:

- Built form variables fail to explain ridership  
- Observed thresholds are **statistical artifacts**  
- System behavior is governed by **non-spatial or network factors**

---

## 9. One-Line Insight

> Kochi is a **“structure-constrained transit system”**, not a morphology-driven one —  
> ridership depends on connectivity and geography, not built form.

# Bengaluru Metro Morphology–Ridership Analysis  
*(Random Forest + PDP/ICE + Elasticity + Thresholds)*

---

## 1. Model Performance & Spatial Signal

| Ring | R² | RMSE | MAE | Interpretation |
|------|----|------|-----|---------------|
| r1 | 0.162 | 0.569 | 0.396 | Moderate signal (localized) |
| r2 | 0.178 | 0.564 | 0.363 | Slightly stronger than r1 |
| r3 | 0.005 | 0.620 | 0.422 | Negligible |

**Key Insight:**  
Bengaluru exhibits a **distributed but weak morphology signal**, with influence extending to **r2**, unlike Hyderabad (r1-dominant).

👉 This suggests a **corridor-based urban structure**, not purely station-centric.

---

## 2. Key Morphological Drivers

Across rings:

- **OSR (Open Space Ratio)** → strongest and most consistent driver  
- **Building Count** → nonlinear, threshold-sensitive density effect  
- **BCR / Mean Height** → secondary structural modifiers  

---

## 3. Threshold Summary

| Ring | Feature | Threshold |
|------|--------|----------|
| r1 | OSR | 0.028 |
| r1 | BCR | 0.694 |
| r1 | Building Count | 0.197 |
| r2 | OSR | 0.035 |
| r2 | Mean Height | 0.386 |
| r2 | Building Count | 0.225 |
| r3 | OSR | 0.037 |
| r3 | Mean Height | 0.422 |
| r3 | BCR | 0.065 |

---

## 4. Ring-wise Interpretation

---

### 🔵 R1 — Immediate Station Area (Controlled Activation Zone)

#### OSR (~0.03)
- Sharp rise followed by plateau  

**Interpretation:**  
Small amounts of open space significantly improve ridership,  
but effect **saturates quickly**.

👉 Indicates:
> Need for **minimum breathing space**, not large open areas

---

#### BCR (~0.69)
- Gradual decline before threshold  
- Slight recovery after  

**Interpretation:**  
High compactness has **mixed effects**,  
suggesting trade-off between accessibility and crowding.

---

#### Building Count (~0.20)
- Positive up to threshold → then flattens  

**Interpretation:**  
Moderate density supports ridership,  
but additional density gives **diminishing returns**

---

**R1 Summary:**

> Optimal morphology =  
> - Moderate density  
> - Small but sufficient open space  
> - Balanced compactness  

---

### 🟡 R2 — Intermediate Zone (Strongest Influence Zone)

#### OSR (~0.035)
- Strong initial gain → flat  

**Interpretation:**  
Open space remains **primary driver even beyond station area**

---

#### Mean Height (~0.39)
- Clear positive jump at threshold  

**Interpretation:**  
Vertical development becomes important in this ring

👉 Suggests:
> Bengaluru demand depends on **corridor-scale vertical intensity**

---

#### Building Count (~0.22)
- Slight positive → then stabilizes  

**Interpretation:**  
Density helps up to moderate levels,  
but does not scale further.

---

**R2 Summary:**

> This is the **effective control zone**, where:
> - Open space + verticality jointly influence ridership

---

### 🔴 R3 — Outer Zone (Weak but Structured Signal)

#### OSR (~0.037)
- Consistent step increase  

#### Mean Height (~0.42)
- Positive but weaker than r2  

#### BCR (~0.065)
- Very low threshold, minimal effect  

**Interpretation:**  
Morphology still shows structure, but:
- Effect size is small  
- Elasticity near zero  

---

**R3 Summary:**

> Outer zone retains weak structural influence,  
but contributes minimally to demand variation.

---

## 5. Cross-Ring Structural Patterns

---

### 1. OSR Dominance Across All Rings

| Ring | OSR Threshold |
|------|-------------|
| r1 | 0.028 |
| r2 | 0.035 |
| r3 | 0.037 |

**Interpretation:**

> Open space is the **most stable and consistent driver**  
across the entire urban structure.

---

### 2. Moderate Density Regime

- Building count thresholds are low (~0.2)  
- No strong negative penalty like Hyderabad  

**Implication:**

> Bengaluru is **not congestion-limited**, but **efficiency-limited**

---

### 3. Verticality Shifts Outward

- r1 → weak role  
- r2 → strong role  
- r3 → weak again  

**Interpretation:**

> Height matters most at **corridor scale**, not at station core

---

### 4. Weak Distance Decay

Unlike Hyderabad:
- Influence does not collapse after r1  
- r2 retains equal or higher importance  

---

## 6. Structural Interpretation

---

### ✔ Corridor-Oriented TOD System

> Ridership is shaped not just at stations,  
but along **continuous urban corridors**

---

### ✔ Open Space as Enabler

> OSR acts as a **universal accessibility regulator**  
rather than a luxury feature

---

### ✔ Controlled Density, Not High Density

> Moderate density works best;  
excess density does not significantly harm, but also does not help

---

## 7. Planning Implications

---

### ✔ What works:
- Maintain **minimum open space thresholds (~3–4%)**
- Promote **mid-rise vertical development in r2**
- Encourage **moderate density clustering**

---

### ❌ What does not work:
- High BCR concentration near stations  
- Over-densification strategies  
- Station-only TOD planning  

---

## 8. Final Synthesis

> Bengaluru Metro operates as a **corridor-based morphological system**, where:

- Open space is the primary regulator of accessibility  
- Vertical development matters more in intermediate zones  
- Density has diminishing returns beyond moderate levels  
- Morphological influence extends beyond station areas  

---

## 9. One-Line Insight

> Bengaluru is a **“corridor TOD city”** —  
ridership is shaped by continuous urban structure, not just station nodes.